In [ ]:
import os
from openai import OpenAI

# The key MUST be inside quotes to be treated as a string
os.environ["OPENAI_API_KEY"] = ""
client = OpenAI()

try:
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": "Say 'Active'"}],
        max_tokens=5
    )
    print(f"✅ Success! Connection established: {response.choices[0].message.content}")
except Exception as e:
    print(f"❌ Error: {e}")

✅ Success! Connection established: Active


In [ ]:
import os
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader, Docx2txtLoader

data_dir = "./Data" 
# for text files text files, you wouldn't have to change your loop at all. You would just add one line to your dictionary: ".txt": TextLoader
loaders_mapping = {
    ".pdf": PyPDFLoader,
    ".docx": Docx2txtLoader
}

all_docs = []

for ext, loader_class in loaders_mapping.items():
    print(f"Scanning for {ext} files...")
    
    current_loader = DirectoryLoader(
        path=data_dir, 
        glob=f"*{ext}", 
        loader_cls=loader_class
    )
    
    # 1. Load the documents once per extension
    docs = current_loader.load()
    
    # 2. Extract and print filenames correctly using Path()
    for doc in docs:
        # Use Path(source) to extract the .name (filename)
        source_path = doc.metadata.get('source', 'Unknown')
        filename = Path(source_path).name
        
    print(f"  --> Loaded content from: {filename}")   
    all_docs.extend(docs)




Scanning for .pdf files...
  --> Loaded content from: HRPolicy.pdf
Scanning for .docx files...
  --> Loaded content from: Leave_policy.docx
--- START OF DOCUMENT: HRPolicy.pdf ---
IIA HR POLICY REVISION 1.1                                                                                 1  
 
 
 
 
 
 
 
 
 
 
 
 
       Indian Industries Association 
 
Human Resource Policy
--- END OF DOCUMENT: HRPolicy.pdf ---

--- START OF DOCUMENT: HRPolicy.pdf ---
IIA HR POLICY REVISION 1.1                                                                                 2  
Preface 
Indian Industries Association (IIA) started as U.P Chapter of NAYE in 1985 was 
renamed as Indian Industries Association in 1992 and registered as a Society. Since 
then, IIA have expanded its membership base and territorial boundaries not only to 
almost all the districts of U.P but outside U.P also. As on date more than 12000 
members are associated with IIA in  various States i.e Uttar Pradesh, Uttarak hand, 
Delhi, H

In [5]:
# Print the full content of each document loaded
for doc in all_docs:
    filename = Path(doc.metadata.get('source', 'Unknown')).name
    print(f"--- START OF DOCUMENT: {filename} ---")
    print(doc.page_content)
    print(f"--- END OF DOCUMENT: {filename} ---\n" + "="*50 + "\n")

--- START OF DOCUMENT: HRPolicy.pdf ---
IIA HR POLICY REVISION 1.1                                                                                 1  
 
 
 
 
 
 
 
 
 
 
 
 
       Indian Industries Association 
 
Human Resource Policy
--- END OF DOCUMENT: HRPolicy.pdf ---

--- START OF DOCUMENT: HRPolicy.pdf ---
IIA HR POLICY REVISION 1.1                                                                                 2  
Preface 
Indian Industries Association (IIA) started as U.P Chapter of NAYE in 1985 was 
renamed as Indian Industries Association in 1992 and registered as a Society. Since 
then, IIA have expanded its membership base and territorial boundaries not only to 
almost all the districts of U.P but outside U.P also. As on date more than 12000 
members are associated with IIA in  various States i.e Uttar Pradesh, Uttarak hand, 
Delhi, Haryana, Gujrat, Maharashtra and Tamil Nadu etc. 
IIA functioning is through the Office Bearers who are owning an enterprise and spend 
some 

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
# 3. Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""]
)

split_docs = text_splitter.split_documents(all_docs)
print(f"\nTotal chunks created: {len(split_docs)}")


Total chunks created: 149


In [11]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

# 1. Initialize Embeddings (Uses the OS environment variable you set)
# Ensure os.environ["OPENAI_API_KEY"] is set in a previous cell!
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 2. Create and Persist Vector Store
# This will create a folder named 'chroma_db_langchain'
vector_db = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings,
    persist_directory="./chroma_db_langchain"
)

print("✅ Success! LangChain Vector Store created and saved to disk.")

✅ Success! LangChain Vector Store created and saved to disk.


In [31]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate


# 1. Define your prompt
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:
{context}

Question: {question}
""")
from langchain_openai import ChatOpenAI

# 1. Initialize the LLM (The "Brain")
# Ensure your OPENAI_API_KEY is set in your environment variables
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# 2. Now define your retriever (ensure vector_db exists!)
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

# 3. NOW you can build the pipe
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 2. Build the "Pipe" chain (No 'langchain.chains' needed!)
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 3. Run it
result = rag_chain.invoke("give me contact details of insurance medi claim ?")
print(result)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://eu.api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://eu.api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://eu.api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://eu.api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Contact details of insurance medi claim: 9762218139


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://eu.api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://eu.api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
